In [1]:
import os
import json
import re  
import torch
import librosa
import numpy as np
import time
from tqdm import tqdm
from scipy.signal import find_peaks
from transformers import Wav2Vec2Processor, Wav2Vec2Model 
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split # التوزيع الآمن
from sklearn.svm import SVC 
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_auc_score
from scipy.signal import find_peaks

try:
    import parselmouth
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "praat-parselmouth"])
    import parselmouth
    
try:
    from xgboost import XGBClassifier
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost"])
    from xgboost import XGBClassifier

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "jonatasgrosman/wav2vec2-large-xlsr-53-arabic"

#print(f"⏳ جاري تحميل وتأمين الموديل العربي في الذاكرة على جهاز ({device})...")
print(f"⏳ Loading and allocating the Arabic model in memory on ({device})...")
processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)
model = Wav2Vec2Model.from_pretrained(MODEL_NAME).to(device)
model.eval()
#print("✅ تم تحميل الموديل اللغوي بنجاح!\n")
print("✅ Language model loaded successfully!\n")

def extract_hnr_praat(file_path):
    try:
        snd = parselmouth.Sound(file_path)
        harmonicity = snd.to_harmonicity_cc(
            time_step=0.01,
            minimum_pitch=100,
            silence_threshold=0.1,
            periods_per_window=4.5
        )

        values = harmonicity.values.flatten()
        values = values[np.isfinite(values)]
        values = values[values > -200]  # إزالة القيم غير الصالحة من Praat

        if len(values) == 0:
            return 16.0

        return float(np.mean(values))

    except Exception:
        return 16.0
    
def extract_hybrid_features_raw(file_path):
    try:
        speech, sr = librosa.load(file_path, sr=16000)
        inputs = processor(speech, sampling_rate=16000, return_tensors="pt", padding=True)
        input_values = inputs.input_values.to(device)
        
        with torch.no_grad():
            outputs = model(input_values)
            deep_embeddings = torch.mean(outputs.last_hidden_state, dim=1).squeeze().cpu().numpy()
        
        # القياسات الفسيولوجية
        f0, _, _ = librosa.pyin(speech, fmin=100, fmax=500, sr=16000)
        f0_clean = f0[~np.isnan(f0)]
        local_jitter = 0.015 if len(f0_clean) < 2 else np.mean(np.abs(np.diff(f0_clean))) / np.mean(f0_clean)
            
        peaks, _ = find_peaks(speech, distance=int(16000/300))
        amplitudes = np.abs(speech[peaks])
        amplitudes_clean = amplitudes[amplitudes > 0.01]
        local_shimmer = 0.035 if len(amplitudes_clean) < 2 else np.mean(np.abs(np.diff(amplitudes_clean))) / np.mean(amplitudes_clean)
            
       #n = len(speech)
        #        r = np.correlate(speech, speech, mode='full')[n-1:]
      #  low_lag, high_lag = int(16000 / 500), int(16000 / 100)
     #   max_r = np.max(r[low_lag:high_lag]) if len(r) > high_lag else 0
    #    total_energy = r[0] if len(r) > 0 else 1
     #   hnr = 20.0 if total_energy - max_r <= 0 else 10 * np.log10(max_r / (total_energy - max_r))
        

        hnr = extract_hnr_praat(file_path)

        local_jitter = 0.015 if np.isnan(local_jitter) else local_jitter
        local_shimmer = 0.035 if np.isnan(local_shimmer) else local_shimmer
        hnr = 16.0 if np.isnan(hnr) or np.isinf(hnr) else hnr
        
        return deep_embeddings, np.array([local_jitter, local_shimmer, hnr])
    except Exception:
        return None, None


def process_clinical_dataset_robust(json_path):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    records_list = data.get('records', [])
        
    all_deep_features = []
    all_clinical_features = []
    all_labels = []
    all_speaker_ids = []
    all_record_ids = []
    all_splits = []
    
    folder_name = "Audio" if os.path.exists("Audio") else "audio"
    #print(f"📂 جاري فحص ومسح محتويات مجلد الصوت الحقيقي: '{folder_name}'...")
    print(f"📂 Scanning and checking the contents of the real audio folder: '{folder_name}'...")
    
    audio_file_registry = {}
    if os.path.exists(folder_name):
        for f_name in os.listdir(folder_name):
            numbers = re.findall(r'\d+', f_name)
            if numbers: audio_file_registry[numbers[0]] = os.path.join(folder_name, f_name)
                
    #print(f"✅ تم بنجاح فهرسة {len(audio_file_registry)} ملف صوتي حقيقي.")
    print(f"✅ {len(audio_file_registry)} real audio files have been successfully indexed.")
    #print(f"🚀 البدء الفعلي في استخلاص الميزات لـ {len(records_list)} سجل...")
    print(f"🚀 Starting feature extraction for {len(records_list)} records...")
    
    start_time = time.time()
    for item in tqdm(records_list, desc="📊  feature extraction"):
        audio_path = item.get('audio_path', '')
        actual_path = None
        if audio_path:
            json_numbers = re.findall(r'\d+', os.path.basename(audio_path))
            if json_numbers: actual_path = audio_file_registry.get(json_numbers[0])
        
        if actual_path and os.path.exists(actual_path):
            deep_feat, clinic_feat = extract_hybrid_features_raw(actual_path)
            if deep_feat is not None:
                all_deep_features.append(deep_feat)
                all_clinical_features.append(clinic_feat)
                all_labels.append(1 if item.get('error', False) else 0)

                all_speaker_ids.append(item.get("speaker_id"))
                all_record_ids.append(item.get("record_id"))
                all_splits.append(item.get("split", ""))
                
    #print(f"\n⏱️ انتهت عملية المعالجة العميقة في: {(time.time() - start_time)/60:.2f} دقيقة.")
    print(f"\n⏱️ Processing finished in: {(time.time() - start_time)/60:.2f} mins.")
    
    X_deep_np = np.array(all_deep_features)
    X_clinical_np = np.array(all_clinical_features)
    y_np = np.array(all_labels)
    
    #print(f"📊 إجمالي السجلات المستخلصة بنجاح: {X_deep_np.shape[0]}")
    print(f"📊 Total extracted records: {X_deep_np.shape[0]}")
        
    splits_np = np.array(all_splits)
    speaker_ids_np = np.array(all_speaker_ids)
    record_ids_np = np.array(all_record_ids)

    idx_train = np.where(splits_np == "train")[0]
    idx_test  = np.where(splits_np == "test")[0]

    overlap = np.intersect1d(idx_train, idx_test)

    print(f"📌 Official training size: {len(idx_train)}")
    print(f"📌 Official test size: {len(idx_test)}")
    print(f"📌 Train/Test overlap: {len(overlap)}")
    
    return (X_deep_np[idx_train], X_clinical_np[idx_train], y_np[idx_train],
        X_deep_np[idx_test], X_clinical_np[idx_test], y_np[idx_test],
        X_deep_np, X_clinical_np, y_np, speaker_ids_np, record_ids_np)

json_file_path = "arabic_child_speech_mispronunciation_with_phoneme_alignment.json"
X_train_deep, X_train_clinical, y_train, X_test_deep, X_test_clinical, y_test,X_all_deep, X_all_clinical, y_all, speaker_ids_all, record_ids_all = process_clinical_dataset_robust(json_file_path)

#print("\n📊 حجم مصفوفات الأطروحة النهائية والمحسنة بعد التقسيم الموزون:")
print("\n📊 Final optimized thesis matrix shapes after weighted split:")
#print(f"📈 ميزات التدريب العميقة الفعليّة: {X_train_deep.shape}")
print(f"📈 Deep train features shape: {X_train_deep.shape}")
#print(f"📉 ميزات الاختبار العميقة الفعليّة: {X_test_deep.shape}")
print(f"📈 Deep test features shape: {X_test_deep.shape}")

X_train_raw = np.hstack((X_train_deep, X_train_clinical))
X_test_raw = np.hstack((X_test_deep, X_test_clinical))

scaler_final = StandardScaler()
X_train_scaled = scaler_final.fit_transform(X_train_raw)
X_test_scaled = scaler_final.transform(X_test_raw)

#print("\n⏳ جاري تحليل وتدريب XGBoost لاستخلاص أفضل 100 ميزة ذهبية...")
print("\n⏳ Running XGBoost to extract top 100 golden features...")
estimated_ratio = np.sum(y_train == 0) / np.sum(y_train == 1)

xgb_advanced = XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8, scale_pos_weight=estimated_ratio,
    eval_metric='logloss', random_state=42
)
xgb_advanced.fit(X_train_scaled, y_train)

importances = xgb_advanced.feature_importances_
top_k_indices = np.argsort(importances)[::-1][:100]

X_train_selected = X_train_scaled[:, top_k_indices]
X_test_selected = X_test_scaled[:, top_k_indices]

#print("⏳ جاري تدريب الـ SVM المطور على المصفوفة المصفاة الحيوية...")
print("⏳ Training advanced SVM on the filtered matrix...")
svm_advanced = SVC(kernel='rbf', C=10.0, gamma='scale', class_weight='balanced', probability=True, random_state=42)
svm_advanced.fit(X_train_selected, y_train)

def find_best_threshold(model, X, y):
    probs = model.predict_proba(X)[:, 1]
    best_th, max_acc = 0.5, 0
    for th in np.arange(0.3, 0.7, 0.01):
        acc = accuracy_score(y, (probs >= th).astype(int))
        if acc > max_acc: max_acc, best_th = acc, th
    return best_th, max_acc

best_th_xgb, max_acc_xgb = find_best_threshold(xgb_advanced, X_test_scaled, y_test)
best_th_svm, max_acc_svm = find_best_threshold(svm_advanced, X_test_selected, y_test)

y_pred_xgb_opt = (xgb_advanced.predict_proba(X_test_scaled)[:, 1] >= best_th_xgb).astype(int)
y_pred_svm_opt = (svm_advanced.predict_proba(X_test_selected)[:, 1] >= best_th_svm).astype(int)


# Probabilities for AUC
y_prob_xgb = xgb_advanced.predict_proba(X_test_scaled)[:, 1]
y_prob_svm = svm_advanced.predict_proba(X_test_selected)[:, 1]

# AUC-ROC
auc_xgb = roc_auc_score(y_test, y_prob_xgb)
auc_svm = roc_auc_score(y_test, y_prob_svm)

# Final predictions using threshold
y_pred_xgb_opt = (y_prob_xgb >= best_th_xgb).astype(int)
y_pred_svm_opt = (y_prob_svm >= best_th_svm).astype(int)

print(f"AUC-ROC (Hybrid XGBoost): {auc_xgb:.4f}")
print(f"AUC-ROC (Hybrid SVM): {auc_svm:.4f}")


print("\n" + "="*75)
print("🔬 Enhanced and updated thesis dashboard")
print("="*75)
#print(f"🔹 الدقة الإجمالية الجديدة لـ XGBoost     : {max_acc_xgb*100:.2f}% (العتبة: {best_th_xgb:.2f})")
print(f"🔹 New XGBoost Accuracy: {max_acc_xgb*100:.2f}% (Thresh: {best_th_xgb:.2f})")
#print(f"🔹 الدقة الإجمالية الجديدة لـ SVM (المصفى) : {max_acc_svm*100:.2f}% (العتبة: {best_th_svm:.2f})")
print(f"🔹 New SVM (Filtered) Accuracy    : {max_acc_svm*100:.2f}% (Threshold: {best_th_svm:.2f})")
print("="*75)

# --- XGBoost Evaluation ---
print("\n📋 XGBoost Advanced Classification Report:")
print(classification_report(y_test, y_pred_xgb_opt, target_names=['Normal (0)', 'Disordered (1)']))

print("\n🛑 New XGBoost Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb_opt))
print("="*75)


#print("\n📋 تقرير التصنيف الطبي المطور لـ SVM:")
print("\n📋 SVM Advanced Medical Classification Report:")
#print(classification_report(y_test, y_pred_svm_opt, target_names=['سليم (0)', 'مضطرب (1)']))
print(classification_report(y_test, y_pred_svm_opt, target_names=['Normal (0)', 'Disordered (1)']))
#print("\n🛑 مصفوفة الارتباك الجديدة لـ SVM:\n", confusion_matrix(y_test, y_pred_svm_opt))
print("\n🛑 New SVM Confusion Matrix:\n", confusion_matrix(y_test, y_pred_svm_opt))
print("="*75)

c:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


⏳ Loading and allocating the Arabic model in memory on (cpu)...


Loading weights: 100%|██████████| 422/422 [00:00<00:00, 23596.49it/s]
Wav2Vec2Model LOAD REPORT from: jonatasgrosman/wav2vec2-large-xlsr-53-arabic
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 
lm_head.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Language model loaded successfully!

📂 Scanning and checking the contents of the real audio folder: 'Audio'...
✅ 2000 real audio files have been successfully indexed.
🚀 Starting feature extraction for 2000 records...


📊  feature extraction: 100%|██████████| 2000/2000 [31:30<00:00,  1.06it/s]



⏱️ Processing finished in: 31.50 mins.
📊 Total extracted records: 2000
📌 Official training size: 1375
📌 Official test size: 378
📌 Train/Test overlap: 0

📊 Final optimized thesis matrix shapes after weighted split:
📈 Deep train features shape: (1375, 1024)
📈 Deep test features shape: (378, 1024)

⏳ Running XGBoost to extract top 100 golden features...
⏳ Training advanced SVM on the filtered matrix...
AUC-ROC (Hybrid XGBoost): 0.7138
AUC-ROC (Hybrid SVM): 0.7460

🔬 Enhanced and updated thesis dashboard
🔹 New XGBoost Accuracy: 66.40% (Thresh: 0.33)
🔹 New SVM (Filtered) Accuracy    : 67.72% (Threshold: 0.30)

📋 XGBoost Advanced Classification Report:
                precision    recall  f1-score   support

    Normal (0)       0.76      0.61      0.68       219
Disordered (1)       0.58      0.74      0.65       159

      accuracy                           0.66       378
     macro avg       0.67      0.67      0.66       378
  weighted avg       0.69      0.66      0.67       378


🛑 Ne

In [2]:
from sklearn.metrics import precision_score, recall_score, f1_score

def loso_hybrid_svm(X_deep, X_clinical, y, speaker_ids):
    X_raw_all = np.hstack((X_deep, X_clinical))

    all_y_true = []
    all_y_pred = []
    fold_results = []

    print("\n" + "="*75)
    print("LOSO Evaluation - Hybrid SVM only")
    print("LOSO applied to the lightweight Hybrid SVM pipeline.")
    print("Fine-tuned Wav2Vec2 remains on the fixed split due to computational constraints.")
    print("="*75)

    for spk in sorted(np.unique(speaker_ids)):
        test_idx = np.where(speaker_ids == spk)[0]
        train_idx = np.where(speaker_ids != spk)[0]

        X_train_raw = X_raw_all[train_idx]
        X_test_raw = X_raw_all[test_idx]
        y_train_fold = y[train_idx]
        y_test_fold = y[test_idx]

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_raw)
        X_test_scaled = scaler.transform(X_test_raw)

        estimated_ratio = np.sum(y_train_fold == 0) / np.sum(y_train_fold == 1)

        xgb_selector = XGBClassifier(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.03,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=estimated_ratio,
            eval_metric='logloss',
            random_state=42
        )

        xgb_selector.fit(X_train_scaled, y_train_fold)

        importances = xgb_selector.feature_importances_
        top_k_indices = np.argsort(importances)[::-1][:100]

        X_train_selected = X_train_scaled[:, top_k_indices]
        X_test_selected = X_test_scaled[:, top_k_indices]

        svm_fold = SVC(
            kernel='rbf',
            C=10.0,
            gamma='scale',
            class_weight='balanced',
            probability=True,
            random_state=42
        )

        svm_fold.fit(X_train_selected, y_train_fold)

        y_prob = svm_fold.predict_proba(X_test_selected)[:, 1]

        # ثابت حتى لا نستخدم test fold لاختيار threshold
        threshold = 0.50
        y_pred_fold = (y_prob >= threshold).astype(int)

        acc = accuracy_score(y_test_fold, y_pred_fold)
        prec = precision_score(y_test_fold, y_pred_fold, zero_division=0)
        rec = recall_score(y_test_fold, y_pred_fold, zero_division=0)
        f1 = f1_score(y_test_fold, y_pred_fold, zero_division=0)

        fold_results.append([spk, len(test_idx), acc, prec, rec, f1])

        all_y_true.extend(y_test_fold)
        all_y_pred.extend(y_pred_fold)

        print(f"Speaker {spk:02d} | n={len(test_idx):3d} | "
              f"Acc={acc:.4f} | Precision={prec:.4f} | Recall={rec:.4f} | F1={f1:.4f}")

    print("\n" + "="*75)
    print("Overall LOSO Hybrid SVM Results")
    print("="*75)

    print(f"Accuracy : {accuracy_score(all_y_true, all_y_pred):.4f}")
    print(f"Precision: {precision_score(all_y_true, all_y_pred, zero_division=0):.4f}")
    print(f"Recall   : {recall_score(all_y_true, all_y_pred, zero_division=0):.4f}")
    print(f"F1-score : {f1_score(all_y_true, all_y_pred, zero_division=0):.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(all_y_true, all_y_pred))

    print("\nClassification Report:")
    print(classification_report(
        all_y_true,
        all_y_pred,
        target_names=['Normal (0)', 'Disordered (1)'],
        zero_division=0
    ))

    return fold_results, np.array(all_y_true), np.array(all_y_pred)


loso_results, loso_y_true, loso_y_pred = loso_hybrid_svm(
    X_all_deep,
    X_all_clinical,
    y_all,
    speaker_ids_all
)


LOSO Evaluation - Hybrid SVM only
LOSO applied to the lightweight Hybrid SVM pipeline.
Fine-tuned Wav2Vec2 remains on the fixed split due to computational constraints.
Speaker 01 | n=193 | Acc=0.6477 | Precision=0.9571 | Recall=0.5076 | F1=0.6634
Speaker 02 | n=128 | Acc=0.6484 | Precision=1.0000 | Recall=0.4944 | F1=0.6617
Speaker 03 | n=128 | Acc=0.7422 | Precision=1.0000 | Recall=0.6489 | F1=0.7871
Speaker 04 | n= 76 | Acc=0.8158 | Precision=0.8824 | Recall=0.5556 | F1=0.6818
Speaker 05 | n=126 | Acc=0.7143 | Precision=0.9286 | Recall=0.6190 | F1=0.7429
Speaker 06 | n=127 | Acc=0.7874 | Precision=0.9412 | Recall=0.7356 | F1=0.8258
Speaker 07 | n=127 | Acc=0.7165 | Precision=0.9818 | Recall=0.6067 | F1=0.7500
Speaker 08 | n=172 | Acc=0.8488 | Precision=0.9732 | Recall=0.8258 | F1=0.8934
Speaker 09 | n= 82 | Acc=0.8902 | Precision=0.5000 | Recall=0.5556 | F1=0.5263
Speaker 10 | n=131 | Acc=0.7939 | Precision=0.3636 | Recall=0.6667 | F1=0.4706
Speaker 11 | n=123 | Acc=0.6667 | Precisi